In [ ]:
import os
import sys
from pathlib import Path

# Locate the repo root as the directory containing this notebook.
# __vsc_ipynb_file__ is injected by VS Code; fall back to cwd for other environments.
try:
    NOTEBOOK_DIR = str(Path(__vsc_ipynb_file__).parent.resolve())
except NameError:
    NOTEBOOK_DIR = str(Path.cwd())

os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

print(f"Working directory: {os.getcwd()}")

In [ ]:
import pandas as pd
from pyspedas import CDAWeb
from l1_pipeline import get_one_day_swmf_input, create_position_file
from l1_combine import create_combined_l1_files

cda = CDAWeb()

In [ ]:
# -----------------------------------------------------------------------
# Date range to process.
# Uncomment the rolling-window block to always run the last N days.
# -----------------------------------------------------------------------

# today = pd.Timestamp.utcnow().strftime('%Y-%m-%d')
# days = pd.date_range(end=today, periods=7).strftime('%Y-%m-%d').tolist()

from plot_l1_may2024 import plot_day

days = pd.date_range(
    start='2024-05-01', end='2024-05-31'
).strftime('%Y-%m-%d').tolist()

start_time = pd.Timestamp.now()

# Seed the window: download the day before day-1 (context only) and day-1.
day_before = (pd.Timestamp(days[0]) -
              pd.Timedelta(days=1)).strftime('%Y-%m-%d')
get_one_day_swmf_input(day_before, cda)
get_one_day_swmf_input(days[0], cda)
create_position_file(days[0], cda)

# Crawling window: download tomorrow, combine today (yesterday + tomorrow
# as context), plot, then advance.  Each day is downloaded exactly once.
for i, day in enumerate(days):
    prev_day = day_before if i == 0 else days[i - 1]

    if i < len(days) - 1:
        next_day = days[i + 1]
        get_one_day_swmf_input(next_day, cda)
        create_position_file(next_day, cda)
    else:
        next_day = None

    create_combined_l1_files(day, prev_day=prev_day, next_day=next_day)
    try:
        plot_day(day)
    except Exception as exc:
        print(f"  Plot error for {day}: {exc}")
    print(f"Completed: {day}")

end_time = pd.Timestamp.now()
print(f"\nDone. Elapsed: {end_time - start_time}")

In [ ]:
# -----------------------------------------------------------------------
# Context plots: each day with ±6 h of the neighbouring days.
# Vertical dashed lines mark midnight boundaries so you can immediately
# spot any jumps between days.  Shaded regions are the context windows.
# Output goes to  plots/with_context/
# Standalone — can be run without running cell 3 first.
# -----------------------------------------------------------------------

import pandas as pd
from plot_l1_may2024 import plot_day_with_context

days = pd.date_range(
    start='2024-05-01', end='2024-05-31'
).strftime('%Y-%m-%d').tolist()

context_start = pd.Timestamp.now()

for day in days:
    try:
        plot_day_with_context(day, context_hours=6,
                              output_dir='plots/with_context')
    except Exception as exc:
        print(f"  Context-plot error for {day}: {exc}")

context_end = pd.Timestamp.now()
print(f"\nContext plots done. Elapsed: {context_end - context_start}")